# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [ ]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets "pillow<11.0" pandas --upgrade
!pip -q install flash-attn --no-build-isolation # 👈 Flash Attention 2를 위한 라이브러리 추가

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [ ]:
# GitHub 레포지토리 클론
# 아래 URL을 본인의 실제 GitHub 레포지토리 주소로 변경해주세요.
GIT_REPO_URL = "https://github.com/username/repository.git"

!git clone $GIT_REPO_URL

In [ ]:
# 압축 해제
# 왼쪽 폴더(📁)에 직접 업로드한 zip 파일의 이름을 아래에 적어주세요.
ZIP_FILE_PATH = "/2026_15_2_ai_DataSet.zip"

!unzip -o $ZIP_FILE_PATH -d "/content/"

# 파일 목록 확인 (경로 확인용)
import os
print("압축 해제 후 파일 목록:", os.listdir('/content/'))

# 라이브러리, 데이터, 설정

In [ ]:
# /content/ 경로의 파일 및 폴더 목록 확인
import os

print("현재 위치의 파일 목록:")
for item in os.listdir('/content/'):
    print("-", item)

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정 (T4)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
DATA_DIR = "/content/2026_15_2_ai_DataSet"
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

# 컬럼명 확인 및 경로 설정
# 'img_path'가 없으면 'id'를 파일명으로 간주하여 경로 생성
img_col = 'img_path' if 'img_path' in train_df.columns else 'id'

train_df['path'] = train_df[img_col].apply(lambda x: os.path.join(DATA_DIR, 'train', x))
test_df['path'] = test_df[img_col].apply(lambda x: os.path.join(DATA_DIR, 'test', x))

# 학습데이터 200개만 추출
# train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

print("Columns:", train_df.columns.tolist())
print("Sample path:", train_df['path'].iloc[0])

# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. 양자화 설정 (A100 권장: bfloat16)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 2. 프로세서 설정 (해상도 유연성 확보)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE * 2,
    trust_remote_code=True,
)

# 사전학습 모델 로드 (Flash Attention 2 적용!)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2", # 👈 속도 및 메모리 최적화
)

# 양자화 모델로 로드 준비
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# 3. LoRA 세팅 (학습 용량 최적화 - 다이어트)
lora_config = LoraConfig(
    r=8,              # 👈 32에서 8로 감소 (학습 속도 향상)
    lora_alpha=16,    # 👈 64에서 16으로 감소
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"], # 👈 전체 선형 레이어에서 어텐션 핵심 레이어로 축소
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
# 모델 지시사항 (역할 부여 및 제약 조건 강화)
SYSTEM_INSTRUCT = (
    "You are an expert visual AI assistant specialized in accurately counting and analyzing objects in images. "
    "Your task is to answer the multiple-choice question based on the provided image. "
    "You MUST output ONLY exactly one lowercase letter (a, b, c, or d). "
    "Do not include any explanations, punctuation, or extra words."
)

# 프롬프트 (구조화 및 주의력 환기)
def build_mc_prompt(question, a, b, c, d):
    return (
        f"질문: {question}\n\n"
        f"[선택지]\n"
        f"a) {a}\n"
        f"b) {b}\n"
        f"c) {c}\n"
        f"d) {d}\n\n"
        "지시사항: 이미지 내부의 객체를 꼼꼼히 확인하고 개수를 세어보세요. "
        "정답에 해당하는 단 하나의 알파벳 소문자(a, b, c, d 중 하나)만 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
from PIL import Image
import torchvision.transforms as transforms

# -----------------------------------------------------
# 1. 데이터 증강 파이프라인 정의 (학습용) - 수정됨 (객체 잘림 방지)
# -----------------------------------------------------
# PIL 이미지 상태에서 바로 적용 가능한 기본 증강들
# 🚨 회전(Rotation) 및 이동/확대(Affine)는 가장자리 객체를 잘라내어
# 라벨(정답 개수)을 훼손할 수 있으므로 제거했습니다.
train_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), # 밝기, 대비 무작위 변경
    transforms.RandomHorizontalFlip(p=0.5),                               # 50% 확률로 좌우 반전 (개수 유지)
])

# -----------------------------------------------------
# 2. 커스텀 데이터셋
# -----------------------------------------------------
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True, transform=None):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        self.transform = transform # 👈 외부에서 증강 기법을 주입받도록 수정

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        # 👈 학습(train) 중이고 transform이 정의되어 있다면 이미지 증강 적용
        if self.train and self.transform:
            img = self.transform(img)

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]

        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# -----------------------------------------------------
# 3. 데이터 콜레이터
# -----------------------------------------------------
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            # 👈 평가(Valid/Test) 시에는 모델이 답변을 생성해야 하므로 add_generation_prompt=True 적용
            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=not self.train
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()

            # 👈 매우 중요한 최적화: 패딩(Padding) 토큰은 학습하지 않도록 -100으로 변경
            # PyTorch의 CrossEntropyLoss는 기본적으로 라벨이 -100인 토큰의 Loss를 계산하지 않습니다.
            labels[enc["attention_mask"] == 0] = -100

            enc["labels"] = labels

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# -----------------------------------------------------
# 🚨 수정 포인트 1: VQAMCDataset 설정
# -----------------------------------------------------
# 학습용: train=True, 앞서 만든 증강(train_transform) 적용
train_ds = VQAMCDataset(train_subset, processor, train=True, transform=train_transform)

# 검증용: Validation Loss 계산을 위해 정답이 필요하므로 train=True 로 설정합니다.
# 평가용이므로 이미지 증강(transform)은 제외합니다.
valid_ds = VQAMCDataset(valid_subset, processor, train=True, transform=None)


# -----------------------------------------------------
# 🚨 수정 포인트 2: DataLoader 및 DataCollator 설정
# -----------------------------------------------------
# 학습용 콜레이터: train=True
train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(processor, train=True),
    num_workers=0
)

# 검증용 콜레이터: Loss 계산을 위해 label을 생성해야 하므로 train=True 로 설정합니다.
valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(processor, train=True),
    num_workers=0
)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
from tqdm.auto import tqdm
import math
import torch
from transformers import get_cosine_schedule_with_warmup

model = model.to(device)

# -----------------------------------------------------
# 🎛️ 하이퍼파라미터 설정
# -----------------------------------------------------
EPOCHS = 3
GRAD_ACCUM = 1            # 👈 배치 사이즈가 8로 커졌으므로 누적 스텝을 1로 변경
LEARNING_RATE = 5e-5      # 학습률
WEIGHT_DECAY = 0.01       # 가중치 감쇠 (과적합 방지)
WARMUP_RATIO = 0.05       # 웜업 비율 (초반 5% 동안 학습률 서서히 증가)

# 옵티마이저
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# 학습 스케줄러 (성능 향상을 위해 Cosine 스케줄러 도입)
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * WARMUP_RATIO),
    num_training_steps=num_training_steps
)

# 최신 PyTorch API 반영
scaler = torch.amp.GradScaler('cuda', enabled=True)

# 최고 성능(Valid Loss 최소값) 기록을 위한 변수 초기화
best_val_loss = float('inf')
global_step = 0

# -----------------------------------------------------
# 🚀 학습 루프 시작
# -----------------------------------------------------
for epoch in range(EPOCHS):
    model.train() # 학습 모드
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        # 최신 PyTorch API 반영
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
            running = 0.0

    # -----------------------------------------------------
    # 🧪 검증(Validation) 루프
    # -----------------------------------------------------
    model.eval() # 평가 모드
    val_loss = 0.0
    val_steps = 0

    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [valid]", unit="batch"):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1

    val_loss_avg = val_loss / val_steps
    print(f"\n[Epoch {epoch+1} 결과] Valid Loss: {val_loss_avg:.4f}")

    # -----------------------------------------------------
    # 💾 최고 성능 모델 저장 로직
    # -----------------------------------------------------
    if val_loss_avg < best_val_loss:
        best_val_loss = val_loss_avg
        SAVE_DIR = "/content/qwen2_5_vl_3b_lora_best"

        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"★ 새로운 최고 성능 갱신! 모델이 안전하게 저장되었습니다. (Loss: {best_val_loss:.4f})\n")
    else:
        print(f"성능이 이전보다 개선되지 않았습니다. (현재 최고 기록: {best_val_loss:.4f})\n")

print(f"🎉 {EPOCHS} 에포크 설정으로 모든 학습 및 평가 과정이 성공적으로 완료되었습니다!")


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
import torchvision.transforms as transforms

# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# -----------------------------------------------------
# 🧪 TTA (Test-Time Augmentation) 증강 기법 정의
# -----------------------------------------------------
tta_transforms = [
    lambda x: x,                                          # 1. 원본 이미지
    transforms.RandomHorizontalFlip(p=1.0),               # 2. 100% 확률로 좌우 반전
    transforms.ColorJitter(brightness=0.3, contrast=0.3), # 3. 밝기 및 대비 조절
]

# 추론을 위해 모든 레이어 고정(Freeze) 및 평가 모드 전환
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference (TTA)", unit="sample"):
    row = test_df.iloc[i]
    orig_img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    sample_preds = []

    # TTA 반복문: 하나의 이미지에 대해 3번의 변형과 예측을 수행합니다.
    for t in tta_transforms:
        img = t(orig_img.copy())

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text}
            ]}
        ]

        # 모델이 스스로 대답을 시작하도록 add_generation_prompt=True 적용
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

        # A100 GPU 추론 속도 극대화를 위한 bfloat16 적용
        with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
            out_ids = model.generate(
                **inputs,
                max_new_tokens=2,
                do_sample=False, # 정답을 무작위성 없이 가장 확률 높은 것으로만 고름(Greedy)
                eos_token_id=processor.tokenizer.eos_token_id
            )

        # 프롬프트 부분을 제외하고 '새롭게 생성된 정답 토큰'만 깔끔하게 잘라내기
        input_len = inputs.input_ids.shape[1]
        generated_ids = out_ids[0][input_len:]

        output_text = processor.decode(generated_ids, skip_special_tokens=True).strip()
        sample_preds.append(extract_choice(output_text))

    # 💡 다수결 투표 (Majority Voting)
    counter = Counter(sample_preds)
    # 동점일 경우 원본 이미지의 결과(sample_preds[0])를 우선시 하기 위해 원본 결과를 먼저 고려하는 로직도 가능하지만,
    # most_common은 기본적으로 먼저 등장한 빈도 높은 값을 반환합니다.
    best_choice = counter.most_common(1)[0][0]
    preds.append(best_choice)

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("🎉 TTA Inference Complete! Saved /content/submission.csv")

### YOLO + Qwen (Two-Stage Pipeline)
1. `ultralytics` 라이브러리를 설치합니다.
2. 이미지에 포함된 객체 목록을 YOLO로 뽑아내어 텍스트 형태의 힌트(`yolo_hint`)로 만듭니다.
3. 해당 힌트를 프롬프트에 주입하여 VLM이 객체 수를 세는 데 참고하도록 합니다.

In [ ]:
!pip -q install ultralytics

In [ ]:
from ultralytics import YOLO
from collections import Counter
from tqdm.auto import tqdm

# 1. 사전학습된 YOLOv8 모델 로드 (COCO 80개 클래스)
# 👈 힌트의 정확도를 높이기 위해 n(nano) 모델에서 m(medium) 모델로 업그레이드
yolo_model = YOLO('yolov8m.pt')

# 2. YOLO 탐지 결과를 힌트 텍스트로 변환하는 함수
def get_yolo_hint(img_path):
    # verbose=False로 불필요한 로그 출력 방지
    results = yolo_model(img_path, verbose=False)
    names = yolo_model.names
    detected = []
    for r in results:
        for c in r.boxes.cls:
            detected.append(names[int(c)])

    counts = Counter(detected)
    if not counts:
        return "탐지된 객체 없음"

    # 예: "person: 2개, bottle: 1개"
    hint = ", ".join([f"{k}: {v}개" for k, v in counts.items()])
    return f"YOLO 탐지 결과: {hint}"

# 3. 데이터프레임에 YOLO 힌트 사전 계산 (학습/추론 속도 저하 방지)
# 모델이 무거워졌으므로 시간이 조금 더 걸릴 수 있습니다.
print("Train 데이터에 YOLO 힌트 캐싱 중...")
tqdm.pandas(desc="Train YOLO")
train_df['yolo_hint'] = train_df['path'].progress_apply(get_yolo_hint)

print("Test 데이터에 YOLO 힌트 캐싱 중...")
tqdm.pandas(desc="Test YOLO")
test_df['yolo_hint'] = test_df['path'].progress_apply(get_yolo_hint)

print("\n✅ YOLO 힌트 생성 완료!")
print("Sample Hint:", train_df['yolo_hint'].iloc[0])


In [ ]:
# 4. 프롬프트 템플릿 업데이트 (YOLO 힌트 반영)
def build_mc_prompt(question, a, b, c, d, yolo_hint=""):
    hint_text = f"\n[사전 탐지 정보]\n{yolo_hint}\n" if yolo_hint else ""
    return (
        f"질문: {question}\n\n"
        f"[선택지]\n"
        f"a) {a}\n"
        f"b) {b}\n"
        f"c) {c}\n"
        f"d) {d}\n"
        f"{hint_text}\n"
        "지시사항: 이미지 내부의 객체를 꼼꼼히 확인하고 개수를 세어보세요. "
        "사전 탐지 정보는 참고만 하시고, 질문에 맞는 객체를 직접 세어 "
        "정답에 해당하는 단 하나의 알파벳 소문자(a, b, c, d 중 하나)만 출력하세요."
    )

# 5. VQAMCDataset 업데이트
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True, transform=None):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        if self.train and self.transform:
            img = self.transform(img)

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])

        yolo_hint = str(row.get("yolo_hint", ""))
        user_text = build_mc_prompt(q, a, b, c, d, yolo_hint)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]

        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터로더 재설정 (업데이트된 데이터셋 적용)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]
train_ds = VQAMCDataset(train_subset, processor, train=True, transform=train_transform)
valid_ds = VQAMCDataset(valid_subset, processor, train=True, transform=None)

# 👈 속도 향상을 위해 batch_size를 8로, num_workers를 4로 증가 (A100 기준)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=DataCollator(processor, train=True), num_workers=4)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False, collate_fn=DataCollator(processor, train=True), num_workers=4)
print("✅ 데이터로더 Batch Size 8, num_workers 4 적용 완료!")


In [ ]:
# 6. 추론 루프 업데이트 (TTA + YOLO 힌트 모두 반영, Batch Inference 최적화)
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference (TTA + YOLO Batch)", unit="sample"):
    row = test_df.iloc[i]
    orig_img = Image.open(row["path"]).convert("RGB")

    # 👈 YOLO 힌트를 프롬프트에 추가
    yolo_hint = row.get("yolo_hint", "")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"], yolo_hint)

    # 1. 3가지 TTA 이미지를 병렬로 처리하기 위해 리스트 생성
    batched_texts = []
    batched_images = []

    for t in tta_transforms:
        img = t(orig_img.copy())
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text}
            ]}
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batched_texts.append(text)
        batched_images.append(img)

    # 2. 배치(Batch) 단위로 프로세서에 입력 (batch_size = 3)
    inputs = processor(text=batched_texts, images=batched_images, return_tensors="pt", padding=True).to(device)

    # 3. 한 번에 3장의 이미지 추론 (Flash Attention 2 자동 적용!)
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        out_ids = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    # 4. 결과 디코딩 및 정답 추출
    input_len = inputs.input_ids.shape[1]
    generated_ids = out_ids[:, input_len:]
    output_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

    sample_preds = [extract_choice(out.strip()) for out in output_texts]

    # 5. 다수결 투표
    counter = Counter(sample_preds)
    best_choice = counter.most_common(1)[0][0]
    preds.append(best_choice)

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission_yolo_tta.csv", index=False)
print("🎉 YOLO + TTA (Batch) Inference Complete! Saved /content/submission_yolo_tta.csv")


In [ ]:
# 모델 응답 예시
print(output_text)